In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Modelling
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Visualisation
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots

## 1. Exploratory Data Analysis

We analyse the **Student Performance Factors** dataset, covering variable structure, missing data, and relationships with the target variable `Exam_Score`. Two analytical tracks are derived based on the missing data findings:

- **Track A (Imputed):** All rows are retained; missing values in three categorical columns are filled by training-set mode imputation.
- **Track B (Dropped):** Rows with any missing values are removed via listwise deletion.

### 1.1 Data Loading

In [ ]:
df = pd.read_csv("../data/StudentPerformanceFactors.csv")

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("Features List:")
print(*df.columns, sep=", ")

### 1.2 Missing Data and Cleaning Strategy

In [ ]:
missing = df.isna().sum()
missing_table = pd.DataFrame({
    "Dtype": df.dtypes,
    "Count": missing,
    "Proportion": (missing / len(df)).round(4),
    "Percentage (%)": (missing / len(df) * 100).round(2)
}).sort_values("Count", ascending=False)

# Display only rows with missing values
display(missing_table[missing_table["Count"] > 0])

All three variables with missing values are categorical, making **mode imputation** a natural choice. As the row-wise summary below shows, only 3.47% of observations have any missing entry, so **listwise deletion** is equally viable with negligible data loss. We therefore proceed with both strategies in parallel.

In [ ]:
# Row-wise (Observation) Missing Data Summary
row_missing_count = df.isna().sum(axis=1)
row_missing_summary = (
    row_missing_count
    .value_counts()
    .sort_index()
    .to_frame(name="Count")
)

row_missing_summary["Proportion"] = (row_missing_summary["Count"] / len(df)).round(4)
row_missing_summary["Percentage (%)"] = (row_missing_summary["Proportion"] * 100).round(2)
row_missing_summary.index.name = "Missing Values per Row"

row_missing_summary

The vast majority of observations (96.53%) contain no missing values, and only 0.09% have more than one missing entry. Dropping rows with missing values therefore retains 96.53% of the data, making listwise deletion a low-cost alternative to imputation.

In [ ]:
# Track A: retain all rows (imputation applied later, within the train split)
df_imputed = df.copy()

# Track B: drop rows with any missing value in the identified columns
df_dropped = df.dropna(subset=['Parental_Education_Level', 'Teacher_Quality', 'Distance_from_Home'])

rows_lost = df.shape[0] - df_dropped.shape[0]
print(f"Track A (Imputed) shape: {df_imputed.shape}")
print(f"Track B (Dropped) shape: {df_dropped.shape}  ({rows_lost} rows removed)")

### 1.3 Variable Overview

In [ ]:
# Create a clean two-column summary table
dtype_summary = df.dtypes.to_frame(name="Data Type")
dtype_summary.index.name = "Variable"

# Sortby type
display(dtype_summary.sort_values("Data Type"))

The dataset contains seven continuous numerical variables — `Hours_Studied`, `Attendance`, `Sleep_Hours`, `Previous_Scores`, `Tutoring_Sessions`, `Physical_Activity`, and `Exam_Score` — with the remaining twelve being categorical.

In [ ]:
fig, ax = subplots()
ax.hist(df["Exam_Score"], bins=20)
ax.set_title("Distribution of Exam Score")
ax.set_xlabel("Exam Score")
ax.set_ylabel("Frequency")

`Exam_Score` is approximately bell-shaped and centred around 67, with a slight right tail from a small number of high-scoring students. The near-symmetric distribution is consistent with a linear regression framework.

In [ ]:
df.describe().round(2)

The mean `Exam_Score` is 67.24 (s.d. 3.89), indicating relatively low dispersion. `Attendance` (s.d. 11.55) and `Hours_Studied` (s.d. 5.99) show considerably more variation, suggesting potential predictive power.

### 1.4 Correlation with Target

Pearson correlations between each numerical predictor and `Exam_Score`, computed separately for Track A and Track B.

In [ ]:
pd.concat([
    df.corr(numeric_only=True)["Exam_Score"].sort_values().rename("Track A (Imputed)"),
    df_dropped.corr(numeric_only=True)["Exam_Score"].sort_values().rename("Track B (Dropped)")
], axis=1)

`Attendance` (~0.58) and `Hours_Studied` (~0.45) are the strongest predictors in both tracks. `Sleep_Hours` and `Physical_Activity` are near zero. Correlation coefficients are nearly identical across tracks, confirming that the small number of dropped rows has negligible impact on the linear signal.

## 2. Data Preprocessing

We now prepare both experimental tracks for modeling. This involves splitting the data, handling missing values for Track A, and encoding categorical variables for both.

In [ ]:
TARGET = "Exam_Score"

### Dataset A: Training-Set Imputation

To ensure the integrity of our generalisation tests, we calculate the mode using only the training data and apply it to the test set.

In [ ]:
X_a = df_imputed.drop(TARGET, axis=1)
y_a = df_imputed[TARGET]

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_a, y_a, test_size=0.2, random_state=42
)

# Mode Imputation (Loop Method; it prevents data leakage)
cat_missing = ["Parental_Education_Level", "Teacher_Quality", "Distance_from_Home"]

for col in cat_missing:
    mode_value = X_train_a[col].mode()[0]      
    X_train_a[col] = X_train_a[col].fillna(mode_value)
    X_test_a[col] = X_test_a[col].fillna(mode_value)

# Categorical Encoding
X_train_a_enc = pd.get_dummies(X_train_a, drop_first=True)
X_test_a_enc = pd.get_dummies(X_test_a, drop_first=True)

# Align train and test so they have the same columns
X_train_a_enc, X_test_a_enc = X_train_a_enc.align(X_test_a_enc, join="left", axis=1, fill_value=0)

### Dataset B: Listwise Deletion

Since Dataset B was already cleaned by dropping rows, we simply proceed with splitting and encoding.

In [ ]:
X_b = df_dropped.drop(TARGET, axis=1)
y_b = df_dropped[TARGET]

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_b, y_b, test_size=0.2, random_state=42
)

# Categorical Encoding
X_train_b_enc = pd.get_dummies(X_train_b, drop_first=True)
X_test_b_enc = pd.get_dummies(X_test_b, drop_first=True)

In [ ]:
# Check:
print(f"Track A columns: {len(X_train_a_enc.columns)}")
print(f"Track B columns: {len(X_train_b_enc.columns)}")

## 3. Modelling

### Fitting Multiple Linear Regression 

We fit Multiple Linear Regression models to both experimental tracks to evaluate if the missing data handling strategy (Imputation vs. Deletion) significantly alters the model's coefficients or predictive performance.

#### Track A: Imputed Dataset

For Track A, missing categorical values were filled using the training-set mode (Loop Method) to preserve the original sample size of 6,607 observations while avoiding data leakage.

In [ ]:
lr_a = LinearRegression()
lr_a.fit(X_train_a_enc, y_train_a)

# Initial Hold-out estimate
# Train-Test Evaluation for Track A
pred_train_a = lr_a.predict(X_train_a_enc)
pred_test_a = lr_a.predict(X_test_a_enc)
rmse_train_a = np.sqrt(mean_squared_error(y_train_a, pred_train_a))
rmse_test_a = np.sqrt(mean_squared_error(y_test_a, pred_test_a))

# Combine the cleaned and encoded training/test sets
X_full_a_enc = pd.concat([X_train_a_enc, X_test_a_enc]).reset_index(drop=True)
y_full_a = pd.concat([y_train_a, y_test_a]).reset_index(drop=True)

# 10-Fold Cross-Validation on combined cleaned data
cv_lr_a = cross_validate(lr_a, X_full_a_enc, y_full_a, cv=10, scoring="neg_root_mean_squared_error")
rmse_cv_a = -cv_lr_a["test_score"].mean()
std_cv_a = cv_lr_a["test_score"].std()

print(f"Track A - Training RMSE: {rmse_train_a:.2f}")
print(f"Track A - Test RMSE: {rmse_test_a:.2f}")

print(f"Track A - Hold-out RMSE: {rmse_test_a:.3f}")
print(f"Track A - CV Mean RMSE: {rmse_cv_a:.3f} (s.d. {std_cv_a:.3f})")


#### Track B: Dropped Dataset (Listwise Deletion)

For Track B, observations containing missing values were removed, resulting in a dataset of 6,378 complete entries. This serves as a "pure data" baseline.

In [ ]:
lr_b = LinearRegression()
lr_b.fit(X_train_b_enc, y_train_b)

# Initial Hold-out estimate
# Train-Test Evaluation for Track B
pred_train_b = lr_b.predict(X_train_b_enc)
pred_test_b = lr_b.predict(X_test_b_enc)
rmse_train_b = np.sqrt(mean_squared_error(y_train_b, pred_train_b))
rmse_test_b = np.sqrt(mean_squared_error(y_test_b, pred_test_b))

# Combine the encoded training/test sets for the dropped track
X_full_b_enc = pd.concat([X_train_b_enc, X_test_b_enc]).reset_index(drop=True)
y_full_b = pd.concat([y_train_b, y_test_b]).reset_index(drop=True)

# 10-Fold Cross-Validation on combined data
cv_lr_b = cross_validate(lr_b, X_full_b_enc, y_full_b, cv=10, scoring="neg_root_mean_squared_error")
rmse_cv_b = -cv_lr_b["test_score"].mean()
std_cv_b = cv_lr_b["test_score"].std()

print(f"Track B - Training RMSE: {rmse_train_b:.2f}")
print(f"Track B - Test RMSE: {rmse_test_b:.2f}")

print(f"Track B - Hold-out RMSE: {rmse_test_b:.3f}")
print(f"Track B - CV Mean RMSE: {rmse_cv_b:.3f} (s.d. {std_cv_b:.3f})")

For both tracks, the training and test RMSE remain closely aligned (e.g., 2.09 vs 1.80 in Track A). This consistency suggests that our models are not overfitting and are successfully capturing the underlying signal within the data.

Track A (Imputed) achieved a lower 10-fold CV RMSE (1.961) compared to Track B (2.049). This suggests that retaining the full sample size through mode imputation preserves a stronger predictive signal.

Track B (Dropped) exhibited significantly lower variance, with a standard deviation of 0.330 compared to 0.569 for Track A. This indicates that listwise deletion, while slightly less accurate, produces a more consistent model across different data partitions.

While Track A offers better raw performance, Track B's higher stability and reliance on pure (non-synthetic) data make it a robust baseline for final model selection.

#### Diagnostic Plots

In [ ]:
fig, ax = subplots(figsize=(6, 6))

ax.scatter(y_test_a, pred_test_a)
ax.set_xlabel("Actual Exam_Score (y_test)")
ax.set_ylabel("Predicted Exam_Score (pred_test)")
ax.set_title("Track A (Imputed): Actual vs. Predicted Performance")

# 45-degree line
minv = min(y_test_a.min(), pred_test_a.min())
maxv = max(y_test_a.max(), pred_test_a.max())
ax.plot([minv, maxv], [minv, maxv], '--')

In [ ]:
fig, ax = subplots(figsize=(6, 6))

ax.scatter(y_test_b, pred_test_b)
ax.set_xlabel("Actual Exam_Score (y_test)")
ax.set_ylabel("Predicted Exam_Score (pred_test)")
ax.set_title("Track B (Dropped): Actual vs. Predicted Performance")

# 45-degree line
minv = min(y_test_b.min(), pred_test_b.min())
maxv = max(y_test_b.max(), pred_test_b.max())
ax.plot([minv, maxv], [minv, maxv], '--')

In [ ]:
resid = y_test_a - pred_test_a

fig, ax = subplots(figsize=(6, 4))
ax.scatter(pred_test_a, resid)
ax.axhline(0, linestyle="--")
ax.set_xlabel("Predicted Exam_Score")
ax.set_ylabel("Residual (Actual - Predicted)")
ax.set_title("Track A (Imputed): Prediction Error Analysis")

In [ ]:
resid = y_test_b - pred_test_b

fig, ax = subplots(figsize=(6, 4))
ax.scatter(pred_test_b, resid)
ax.axhline(0, linestyle="--")
ax.set_xlabel("Predicted Exam_Score")
ax.set_ylabel("Residual (Actual - Predicted)")
ax.set_title("Track B (Dropped): Prediction Error Analysis")

In both tracks, the residual and "Actual vs. Predicted" plots show significant deviation for high exam scores. This confirms that the model systemically underpredicts top-performing students.

The diagnostic plots suggest that while the model performs well for mid-range scores, it tends to underpredict students with very high exam scores. This indicates potential model bias in the upper tail.

## 4. Tree-based Models


### Fitting Decision Tree Regressor

To capture potential non-linear relationships and feature interactions that a linear model may miss, we next fit a decision tree regressor on both Track A (imputed) and Track B (dropped), mirroring the structure of the linear regression section.

#### Track A (Imputed): Constrained tree (max_depth = 3)

We begin with a shallow tree to control model complexity and reduce overfitting.

In [ ]:
tree_a_depth3 = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_a_depth3.fit(X_train_a_enc, y_train_a)

pred_train_a_tree3 = tree_a_depth3.predict(X_train_a_enc)
pred_test_a_tree3  = tree_a_depth3.predict(X_test_a_enc)

rmse_train_a_tree3 = np.sqrt(mean_squared_error(y_train_a, pred_train_a_tree3))
rmse_test_a_tree3  = np.sqrt(mean_squared_error(y_test_a,  pred_test_a_tree3))

print(f"Track A - Training RMSE: {rmse_train_a_tree3:.2f}")
print(f"Track A - Test RMSE:     {rmse_test_a_tree3:.2f}")

#### Track B (Dropped): Constrained tree (max_depth = 3)

In [ ]:
tree_b_depth3 = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_b_depth3.fit(X_train_b_enc, y_train_b)

pred_train_b_tree3 = tree_b_depth3.predict(X_train_b_enc)
pred_test_b_tree3  = tree_b_depth3.predict(X_test_b_enc)

rmse_train_b_tree3 = np.sqrt(mean_squared_error(y_train_b, pred_train_b_tree3))
rmse_test_b_tree3  = np.sqrt(mean_squared_error(y_test_b,  pred_test_b_tree3))

print(f"Track B - Training RMSE: {rmse_train_b_tree3:.2f}")
print(f"Track B - Test RMSE:     {rmse_test_b_tree3:.2f}")

In both tracks, the training and test RMSE for the constrained tree are closely aligned, indicating no severe overfitting. However, the test RMSE exceeds that of the linear regression model, suggesting the depth-3 tree is underfitting relative to the linear model.

#### Track A (Imputed): Unconstrained tree

We now remove the depth restriction to illustrate the high-variance behaviour of fully grown decision trees.

In [ ]:
tree_a_full = DecisionTreeRegressor(random_state=42)
tree_a_full.fit(X_train_a_enc, y_train_a)

pred_train_a_full = tree_a_full.predict(X_train_a_enc)
pred_test_a_full  = tree_a_full.predict(X_test_a_enc)

rmse_train_a_full = np.sqrt(mean_squared_error(y_train_a, pred_train_a_full))
rmse_test_a_full  = np.sqrt(mean_squared_error(y_test_a,  pred_test_a_full))

print(f"Track A - Training RMSE: {rmse_train_a_full:.2f}")
print(f"Track A - Test RMSE:     {rmse_test_a_full:.2f}")

#### Track B (Dropped): Unconstrained tree

In [ ]:
tree_b_full = DecisionTreeRegressor(random_state=42)
tree_b_full.fit(X_train_b_enc, y_train_b)

pred_train_b_full = tree_b_full.predict(X_train_b_enc)
pred_test_b_full  = tree_b_full.predict(X_test_b_enc)

rmse_train_b_full = np.sqrt(mean_squared_error(y_train_b, pred_train_b_full))
rmse_test_b_full  = np.sqrt(mean_squared_error(y_test_b,  pred_test_b_full))

print(f"Track B - Training RMSE: {rmse_train_b_full:.2f}")
print(f"Track B - Test RMSE:     {rmse_test_b_full:.2f}")

In both tracks, the unrestricted tree achieves zero training error, indicating complete memorisation of the training data. The test RMSE increases substantially relative to the constrained tree, demonstrating severe overfitting. This highlights the high-variance nature of fully grown decision trees.

#### 10-Fold Cross-Validation

To obtain a more reliable estimate of generalisation performance, we perform 10-fold cross-validation on the full dataset.

In [ ]:
cv = KFold(n_splits=10, shuffle=True, random_state=42)

# Track A: depth=3
cv_tree_a_depth3 = cross_validate(
    DecisionTreeRegressor(max_depth=3, random_state=42),
    X_full_a_enc, y_full_a, cv=cv, scoring="neg_root_mean_squared_error"
)
rmse_cv_a_tree3 = -cv_tree_a_depth3["test_score"].mean()
std_cv_a_tree3  =  cv_tree_a_depth3["test_score"].std()

# Track A: full tree
cv_tree_a_full = cross_validate(
    DecisionTreeRegressor(random_state=42),
    X_full_a_enc, y_full_a, cv=cv, scoring="neg_root_mean_squared_error"
)
rmse_cv_a_full = -cv_tree_a_full["test_score"].mean()
std_cv_a_full  =  cv_tree_a_full["test_score"].std()

# Track B: depth=3
cv_tree_b_depth3 = cross_validate(
    DecisionTreeRegressor(max_depth=3, random_state=42),
    X_full_b_enc, y_full_b, cv=cv, scoring="neg_root_mean_squared_error"
)
rmse_cv_b_tree3 = -cv_tree_b_depth3["test_score"].mean()
std_cv_b_tree3  =  cv_tree_b_depth3["test_score"].std()

# Track B: full tree
cv_tree_b_full = cross_validate(
    DecisionTreeRegressor(random_state=42),
    X_full_b_enc, y_full_b, cv=cv, scoring="neg_root_mean_squared_error"
)
rmse_cv_b_full = -cv_tree_b_full["test_score"].mean()
std_cv_b_full  =  cv_tree_b_full["test_score"].std()

print(f"Track A - DT (depth=3) CV RMSE: {rmse_cv_a_tree3:.3f} (s.d. {std_cv_a_tree3:.3f})")
print(f"Track A - DT (full)    CV RMSE: {rmse_cv_a_full:.3f} (s.d. {std_cv_a_full:.3f})")
print(f"Track B - DT (depth=3) CV RMSE: {rmse_cv_b_tree3:.3f} (s.d. {std_cv_b_tree3:.3f})")
print(f"Track B - DT (full)    CV RMSE: {rmse_cv_b_full:.3f} (s.d. {std_cv_b_full:.3f})")

Across both tracks, the 10-fold CV confirms that the decision tree (depth=3) underperforms relative to linear regression, and the full tree exhibits high variance. The CV RMSE provides a more reliable generalisation estimate than a single hold-out split.

## 5. Model Comparison

### Model Comparison Table

In [ ]:
comparison_df = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree (depth=3)",
        "Decision Tree (full)",
    ],
    "Track A: CV RMSE": [rmse_cv_a,       rmse_cv_a_tree3, rmse_cv_a_full],
    "Track A: CV Std":  [std_cv_a,         std_cv_a_tree3,  std_cv_a_full],
    "Track B: CV RMSE": [rmse_cv_b,       rmse_cv_b_tree3, rmse_cv_b_full],
    "Track B: CV Std":  [std_cv_b,         std_cv_b_tree3,  std_cv_b_full],
})

comparison_df.style.format({
    "Track A: CV RMSE": "{:.3f}",
    "Track A: CV Std":  "{:.3f}",
    "Track B: CV RMSE": "{:.3f}",
    "Track B: CV Std":  "{:.3f}",
})

Across both tracks, linear regression achieves the lowest CV RMSE, confirming it as the strongest model for this dataset. The depth-3 tree underfits, while the full tree severely overfits as evidenced by its high CV RMSE and large standard deviation.

Track A (imputed) consistently yields lower CV RMSE than Track B (dropped), suggesting that retaining the full sample size through mode imputation preserves predictive signal. Track B, however, exhibits smaller standard deviations across models, reflecting greater stability from using only complete observations.

These results illustrate the bias–variance trade-off: greater model flexibility reduces bias but substantially increases variance, leading to poorer generalisation in this case. The comparison across both tracks also demonstrates that missing data handling strategy has a modest but consistent effect on model performance.

to do list for now: 
- drop the missing values instead of imputation (can give an argument about it since i have done imputation already)
- find the depth of the tree regression model that doesnt underfit (depth = 3) of overfit (depth = unrestricted) (the tutor suggested using RMSE vs depth graph but i dont know... is there anything about this in the intro to stat learning with python book?)
- average across the cross validation table instead of the single split test rmse in the Model Comparison table subsection, include the test rmse and the variance (prefer models with less precise and small variance than precise but large variance)
- fit random forest
- any other model?